# 02 — Préparation : découpage Train / Validation / Test

**Entrée :** `data/clean/processed_data.csv` (données nettoyées dans `01_exploration.ipynb`).

**Sortie :** deux versions des jeux, sauvegardées dans `data/clean/` :
- `chronologique/` : découpage selon le temps (**version principale**) ;
- `aleatoire/` : découpage aléatoire stratifié (**version de comparaison**).

Dans les deux cas : 70 % des lignes pour l'entraînement (train), 15 % pour la validation (val) et 15 % pour le test.

## Découpage selon le temps (version principale)

Le fichier est trié par date (de mai 2008 à novembre 2010). On coupe donc **dans l'ordre des lignes, sans mélange** : le train contient le passé, la validation la période suivante et le test la plus récente.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
from pathlib import Path

In [2]:
# Chargement de données préparées et creation d'une copie
df =  pd.read_csv("../data/clean/processed_data.csv")
df_copy = df.copy()

In [3]:
# Fonction pour découper le dataframe en respectant l'ordre chronologique
def split_chronologique(df_copy, prop_train=0.7, prop_val=0.15):
    """ Découpe df_copy en train valid et test sans mélanger les données, en respectant l'ordre chronologique."""

    train, reste = train_test_split(df_copy, train_size=prop_train, shuffle=False)
    val, test = train_test_split(reste, train_size=prop_val/(1-prop_train), shuffle=False)

    return train, val, test

**Deux découpages sont préparés :**

1. **Chronologique (principal).** L'agent devra prédire pour des clients **futurs**, pas pour une période déjà connue. Entraîner sur le passé et tester sur le futur reproduit cette situation.
2. **Aléatoire stratifié (comparaison).** Chaque jeu garde la même proportion de « yes » (11,3 %). En comparant ses résultats à ceux du split chronologique, on mesure si le modèle dépend de l'époque. Il n'est **pas** utilisé pour choisir les modèles.

**À noter :** avec le split chronologique, le taux de « yes » diffère d'un jeu à l'autre (train 5,6 %, val 10,6 %, test 38,4 %). Ce n'est pas une erreur : le taux de souscription augmente avec le temps.

In [4]:
#Vérifier la fonction split_chronologique, aucune ligne perdues, les indices sont bien chronologiques et la moyenne de y_num est cohérente entre les trois parties.
train, val, test = split_chronologique(df_copy)

print("longeur de chauqe partie :", len(train), len(val), len(test))
#Vérifier perde de lignes
print("longeur totale :", len(train) + len(val) + len(test) == len(df_copy))
#Vérifier l'ordre
print("indices chronologiques :", train.index.max() < val.index.min() < test.index.min())

for nom, d in [("train", train), ("val", val), ("test", test)]:
    print(nom, len(d), round(d["y_num"].mean(), 3))

longeur de chauqe partie : 28823 6176 6177
longeur totale : True
indices chronologiques : True
train 28823 0.056
val 6176 0.106
test 6177 0.384


In [5]:
# vérifier qu’il n’y a pas de 2010 dans le train
mois_num = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
            "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
m = df["month"].map(mois_num)
annee = 2008 + (m < m.shift()).cumsum()      # tableau séparé, pas une colonne de df

for nom, d in [("train", train), ("val", val), ("test", test)]:
    print(nom, annee.loc[d.index].value_counts().sort_index().to_dict())

train {2008: 27682, 2009: 1141}
val {2009: 6176}
test {2009: 4119, 2010: 2058}


In [6]:
# Sauvegarde des trois parties dans le dossier clean/chronologique
dossier = Path("../data/clean/chronologique")
dossier.mkdir(parents=True, exist_ok=True)
train.to_csv(dossier / "train.csv", index=False)
val.to_csv(dossier / "val.csv", index=False)
test.to_csv(dossier / "test.csv", index=False)

In [7]:
# Split aléatoire pour comparaison
def split_aleatoire(df_copy, cible="y_num", prop_train=0.7, prop_val=0.15, seed=42):
    """ Découpe df_copy en train valid et test en mélangeant les données."""
    train, reste = train_test_split(df_copy, train_size=prop_train, random_state=seed, stratify=df_copy[cible])
    val, test = train_test_split(reste, train_size=prop_val/(1-prop_train), random_state=seed, stratify=reste[cible])

    return train, val, test

In [8]:
# Vérifier la fonction split_aléatoire, aucune ligne perdues, les indices sont bien mélangés et la moyenne de y_num est cohérente entre les trois parties.
train_a, val_a, test_a = split_aleatoire(df_copy)

print(len(train_a), len(val_a), len(test_a))     # 28823 6176 6177
for nom, d in [("train", train_a), ("val", val_a), ("test", test_a)]:
    print(nom, round(d["y_num"].mean(), 3))      # 0.113 pour les trois

28823 6176 6177
train 0.113
val 0.113
test 0.113


In [9]:
# Sauvegarde des trois parties dans le dossier clean/aleatoire
dossier = Path("../data/clean/aleatoire")
dossier.mkdir(parents=True, exist_ok=True)
train_a.to_csv(dossier / "train.csv", index=False)
val_a.to_csv(dossier / "val.csv", index=False)
test_a.to_csv(dossier / "test.csv", index=False)